## **Arabic Named Entity Recognition**
Named entity recognition (NER) is the task of finding spans of text that constitute proper names and labeling the type of such entities.
As we discussed in the lecture, this task can be formulated as a sequence labeling problem, e.g., using the BIO tagging scheme.
Your task is to build a NER tagger for Arabic using Conditional Random Fields (CRFs) and explore different sets of features to achieve improvement over the baseline tagger.
Arabic is a morphologically rich language, which poses certain challenges to NLP.
We use ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).
Since ANERcorp only has train and test splits, we take 10% of the train as a development set.


## **The Task**
Your task is to improve the feature function (`word2features()`).
Currently, the baseline model only takes into account previous and next words.
You may use external tools to obtain useful features such as POS and morphological tags.
You will also need to submit a report describing your method, result, and analysis (1-2 pages).

## **Data Preparation**

In [1]:
%pip install scikit-learn seqeval
%pip install python-crfsuite
!test -f ANERcorp-CamelLabSplits.zip || wget https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
!unzip -o ANERcorp-CamelLabSplits.zip -x "__MACOSX/*"

/Users/hajackson/venvs/nlp805/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


/Users/hajackson/venvs/nlp805/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


Archive:  ANERcorp-CamelLabSplits.zip
  inflating: ANERcorp-CamelLabSplits/ANERCorp_Benajiba.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt  
  inflating: ANERcorp-CamelLabSplits/README.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt  


In [2]:
from sklearn.model_selection import train_test_split

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

## **Feature Engineering**
Here, your task is to design a function that tranforms each word into a set of features.
You may want to use existing tools to get POS, morphological features, etc.
The current model gives `0.70` in $F_1$ on the dev set.

In [3]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        'word': word
    }

    if i > 0:
        prev_word = sent[i-1][0]

        features.update({
            '-1:word': prev_word
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        next_word = sent[i+1][0]

        features.update({
            '+1:word': next_word,
        })
    else:
        features['EOS'] = True

    return features

Extract the features from the data and train your model.

In [4]:
import pycrfsuite


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

def sent2tokens(sent):
    return [token for token, label in sent]


X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_dev = [sent2features(s) for s in dev_sents]
y_dev = [sent2labels(s) for s in dev_sents]

# create pycrfsuite.Trainer instance
trainer = pycrfsuite.Trainer(verbose=False)

# load the training data
for xseq, yseq in zip(X_train, y_train):
    trainer.append(xseq, yseq)

# training parameteres
trainer.set_params({
    'c1': 1.0,   # coefficient for L1 penalty
    'c2': 1e-3,  # coefficient for L2 penalty
    'max_iterations': 50,  # stop earlier

    # include transitions that are possible, but not observed
    'feature.possible_transitions': True
})

trainer.train('arabic-ner.crfsuite')

## **Evaluation**

Evaluate your model on the development set and the test set.

**Note**: You may run evaluation on the test set **ONLY ONCE** at the end. You should not use the test set to check how well your model is currently performing and modify the model accordingly.
Use the dev set, or create a tune set for this purpose.

In [5]:
# instantiate the trained tagger
tagger = pycrfsuite.Tagger()
tagger.open('arabic-ner.crfsuite')

# tag sentences in the dev set
y_pred_dev = [tagger.tag(xseq) for xseq in X_dev]